# **Predictive Maintenance of Aircraft Engines Using GRU Neural Networks**

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [10]:
cols = ['id','cycle','setting1','setting2','setting3'] + [f's{i}' for i in range(1,22)]

train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
rul_truth = pd.read_csv('RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])

train.head()


,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


# **COMPUTE RUL**

# New Section

In [11]:
max_cycles = train.groupby('id')['cycle'].max().reset_index()
max_cycles.columns = ['id','max_cycle']

train = train.merge(max_cycles, on='id')
train['RUL'] = train['max_cycle'] - train['cycle']
train.drop('max_cycle', axis=1, inplace=True)

train[['id','cycle','RUL']].head()


,id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187


# **CREATE FAILURE TABLE**

In [12]:
train['failure'] = np.where(train['RUL'] <= 30, 1, 0)

train[['id','cycle','RUL','failure']].tail()


,id,cycle,RUL,failure
20626,100,196,4,1
20627,100,197,3,1
20628,100,198,2,1
20629,100,199,1,1
20630,100,200,0,1


# NORMALIZE SENSORS

In [13]:
features = [f's{i}' for i in range(1,22)]

scaler = MinMaxScaler()
train[features] = scaler.fit_transform(train[features])

train[features].head()


,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,0.0,0.183735,0.406802,0.309757,0.0,1.0,0.726248,0.242424,0.109755,0.0,...,0.633262,0.205882,0.199608,0.363986,0.0,0.333333,0.0,0.0,0.713178,0.724662
1,0.0,0.283133,0.453019,0.352633,0.0,1.0,0.628019,0.212121,0.100242,0.0,...,0.765458,0.279412,0.162813,0.411312,0.0,0.333333,0.0,0.0,0.666667,0.731014
2,0.0,0.343373,0.369523,0.370527,0.0,1.0,0.710145,0.272727,0.140043,0.0,...,0.795309,0.220588,0.171793,0.357445,0.0,0.166667,0.0,0.0,0.627907,0.621375
3,0.0,0.343373,0.256159,0.331195,0.0,1.0,0.740741,0.318182,0.124518,0.0,...,0.889126,0.294118,0.174889,0.166603,0.0,0.333333,0.0,0.0,0.573643,0.662386
4,0.0,0.349398,0.257467,0.404625,0.0,1.0,0.668277,0.242424,0.149960,0.0,...,0.746269,0.235294,0.174734,0.402078,0.0,0.416667,0.0,0.0,0.589147,0.704502


In [14]:
sequence_length = 45
X = []
y = []

for engine_id in train['id'].unique():
    engine_data = train[train['id'] == engine_id]
    engine_data = engine_data.sort_values('cycle')

    for i in range(len(engine_data) - sequence_length):
        seq = engine_data.iloc[i:i+sequence_length][features].values
        label = engine_data.iloc[i+sequence_length]['failure']
        X.append(seq)
        y.append(label)

X = np.array(X)
y = np.array(y)

X.shape, y.shape


((16131, 45, 21), (16131,))

# **LSTM MODEL**

In [15]:
#train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)


In [16]:
#build lstm model
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(45,21)))
model.add(Dropout(0.2))

model.add(LSTM(32))
model.add(Dropout(0.2))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 45, 64)         │        22,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 45, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,465 (134.63 KB)

 Trainable params: 34,465 (134.63 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
#train the model
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=200,
    validation_split=0.1
)


Epoch 1/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 13s 158ms/step - accuracy: 0.8039 - loss: 0.4858 - val_accuracy: 0.9233 - val_loss: 0.2061
Epoch 2/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 126ms/step - accuracy: 0.9333 - loss: 0.1823 - val_accuracy: 0.9287 - val_loss: 0.1744
Epoch 3/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 112ms/step - accuracy: 0.9400 - loss: 0.1435 - val_accuracy: 0.9466 - val_loss: 0.1257
Epoch 4/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 140ms/step - accuracy: 0.9572 - loss: 0.1072 - val_accuracy: 0.9582 - val_loss: 0.0920
Epoch 5/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - accuracy: 0.9601 - loss: 0.0947 - val_accuracy: 0.9651 - val_loss: 0.0841
Epoch 6/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 141ms/step - accuracy: 0.9676 - loss: 0.0821 - val_accuracy: 0.9574 - val_loss: 0.0940
Epoch 7/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 138ms/step - accuracy: 0.9633 - loss: 0.0889 - val_accuracy: 0.9682 - val_loss: 0.0748
Epoch 8/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 125ms/step - accuracy: 0.9667 - loss: 0.0799 - val_accuracy: 

In [18]:
#evaluate performance
pred = (model.predict(X_test) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))


101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
Accuracy: 0.9640533002788968
Precision: 0.8637724550898204
Recall: 0.9584717607973422
F1 Score: 0.9086614173228347


# BUILD GRU MODEL


In [19]:
from tensorflow.keras.layers import GRU

gru_model = Sequential()

gru_model.add(GRU(64, return_sequences=True, input_shape=(45,21)))
gru_model.add(Dropout(0.2))

gru_model.add(GRU(32))
gru_model.add(Dropout(0.2))

gru_model.add(Dense(1, activation='sigmoid'))

gru_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

gru_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 45, 64)         │        16,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 45, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,145 (102.13 KB)

 Trainable params: 26,145 (102.13 KB)

 Non-trainable params: 0 (0.00 B)

train GRU

In [20]:
history_gru = gru_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=200,
    validation_split=0.1
)


Epoch 1/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 14s 162ms/step - accuracy: 0.7376 - loss: 0.5009 - val_accuracy: 0.9427 - val_loss: 0.1426
Epoch 2/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 167ms/step - accuracy: 0.9382 - loss: 0.1462 - val_accuracy: 0.9450 - val_loss: 0.1324
Epoch 3/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 129ms/step - accuracy: 0.9429 - loss: 0.1416 - val_accuracy: 0.9481 - val_loss: 0.1275
Epoch 4/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 166ms/step - accuracy: 0.9487 - loss: 0.1199 - val_accuracy: 0.9473 - val_loss: 0.1328
Epoch 5/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 157ms/step - accuracy: 0.9585 - loss: 0.1055 - val_accuracy: 0.9659 - val_loss: 0.0733
Epoch 6/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 137ms/step - accuracy: 0.9744 - loss: 0.0648 - val_accuracy: 0.9566 - val_loss: 0.0960
Epoch 7/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 148ms/step - accuracy: 0.9677 - loss: 0.0728 - val_accuracy: 0.9682 - val_loss: 0.0687
Epoch 8/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 153ms/step - accuracy: 0.9751 - loss: 0.0620 - val_accuracy:

In [21]:
#evaluating gru
pred_gru = (gru_model.predict(X_test) > 0.5).astype(int)

print("GRU Accuracy:", accuracy_score(y_test, pred_gru))
print("GRU Precision:", precision_score(y_test, pred_gru))
print("GRU Recall:", recall_score(y_test, pred_gru))
print("GRU F1 Score:", f1_score(y_test, pred_gru))


101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step
GRU Accuracy: 0.9748992872637124
GRU Precision: 0.9610619469026549
GRU Recall: 0.9019933554817275
GRU F1 Score: 0.9305912596401028


# **PREDICTION FUNCTION**

In [24]:
def predict_engine_health(engine_45_cycles_df):
    engine_45_cycles_df = engine_45_cycles_df[features]   # keep only sensors
    engine_45_cycles_df = scaler.transform(engine_45_cycles_df)  # now numpy array
    engine_45_cycles_df = np.expand_dims(engine_45_cycles_df, axis=0)

    prob = gru_model.predict(engine_45_cycles_df)[0][0]

    if prob > 0.5:
        return f"⚠️ FAILURE RISK (Probability = {prob:.2f})"
    else:
        return f"✅ SAFE (Probability = {prob:.2f})"



In [25]:
engine100 = train[train['id']==100].tail(45)
print(predict_engine_health(engine100))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
✅ SAFE (Probability = 0.05)
